# 21a_gap_cls_selected_block_pdf

Create a qualitative PDF to compare the currently most relevant GAP and CLS settings.

Per image page:
- GAP / mean pooling, best intermediate block (`-10`, resolved block 2)
- GAP / mean pooling, last block (`-1`, resolved block 11)
- CLS token pooling, best intermediate block (`-6`, resolved block 6)
- CLS token pooling, last block (`-1`, resolved block 11)

Input CSV:
- `data/HAM10000/ham_test_cam_all_with_masks_grouped_by_class.csv`

Default selection:
- first `N_PER_CLASS = 10` images per class.

Use this notebook to visually decide which pooling and block setting is most useful for the clinician slide deck.

In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

# =============================================================================
# 1. MAIN PARAMETERS TO MODIFY
# =============================================================================

QUAL_SEED = 42
N_PER_CLASS = 20          # Change this later, e.g. 5, 10, 20
DRY_RUN = False

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

SOURCE_CSV = HAM_ROOT / "ham_test_cam_all_with_masks_grouped_by_class.csv"
SELECTED_CSV = HAM_ROOT / f"ham_test_cam_first{N_PER_CLASS}_per_class_gap_cls_block_pdf.csv"

OUT_ROOT = REPO_ROOT / "outputs" / f"gap_cls_selected_blocks_pdf_first{N_PER_CLASS}_per_class_seed{QUAL_SEED}"
PANEL_ROOT = OUT_ROOT / "panels"
PDF_OUT = OUT_ROOT / f"gap_cls_selected_blocks_first{N_PER_CLASS}_per_class_seed{QUAL_SEED}.pdf"
CONFIG_OUT = OUT_ROOT / f"gap_cls_selected_blocks_config_first{N_PER_CLASS}_per_class_seed{QUAL_SEED}.json"

GT_COL = "gt_label"
CLASS_ARGS = ["--class_preset", "ham"]
COMPARE_ARGS = [
    "--compare_mode", "gt_topk_non_target",
    "--topk_compare", "1",
]

# ViT Base has 12 blocks:
# -1  -> resolved block 11 = last block
# -6  -> resolved block 6
# -10 -> resolved block 2
SCENARIOS = [
    {
        "name": "GAP / mean pooling — block -10",
        "short_name": "gap_block_minus10",
        "checkpoint": REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-HA075.pth",
        "checkpoint_model_type": "panderm",
        "target_block_index": -10,
    },
    {
        "name": "GAP / mean pooling — last block",
        "short_name": "gap_last_block",
        "checkpoint": REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-HA075.pth",
        "checkpoint_model_type": "panderm",
        "target_block_index": -1,
    },
    # {
    #     "name": "CLS token pooling — block -6",
    #     "short_name": "cls_block_minus6",
    #     "checkpoint": REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-HA075-cls.pth",
    #     "checkpoint_model_type": "panderm",
    #     "target_block_index": -6,
    # },
    # {
    #     "name": "CLS token pooling — last block",
    #     "short_name": "cls_last_block",
    #     "checkpoint": REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-HA075-cls.pth",
    #     "checkpoint_model_type": "panderm",
    #     "target_block_index": -1,
    # },
    {
        "name": "GAP 2 / mean pooling — block -10",
        "short_name": "gap_block_minus10",
        "checkpoint": REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-HA10-gap.pth",
        "checkpoint_model_type": "panderm",
        "target_block_index": -10,
    },
    {
        "name": "GAP 2 / mean pooling — last block",
        "short_name": "gap_last_block",
        "checkpoint": REPO_ROOT / "external" / "checkpoints3" / "checkpoint-best-HA10-gap.pth",
        "checkpoint_model_type": "panderm",
        "target_block_index": -1,
    },
]

OUT_ROOT.mkdir(parents=True, exist_ok=True)
PANEL_ROOT.mkdir(parents=True, exist_ok=True)

print("SOURCE_CSV:", SOURCE_CSV)
print("SELECTED_CSV:", SELECTED_CSV)
print("OUT_ROOT:", OUT_ROOT)
print("PDF_OUT:", PDF_OUT)
print("N_PER_CLASS:", N_PER_CLASS)

for s in SCENARIOS:
    print("\n", s["name"])
    print("  checkpoint:", s["checkpoint"])
    print("  target_block_index:", s["target_block_index"])
    if not Path(s["checkpoint"]).exists():
        print("  [WARN] missing checkpoint")


SOURCE_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_all_with_masks_grouped_by_class.csv
SELECTED_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_first20_per_class_gap_cls_block_pdf.csv
OUT_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/gap_cls_selected_blocks_pdf_first20_per_class_seed42
PDF_OUT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/gap_cls_selected_blocks_pdf_first20_per_class_seed42/gap_cls_selected_blocks_first20_per_class_seed42.pdf
N_PER_CLASS: 20

 GAP / mean pooling — block -10
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-HA075.pth
  target_block_index: -10

 GAP / mean pooling — last block
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-HA075.pth
  target_block_index: -1

 GAP 2 / mean pooling — block -10
  checkpoint: /Users/choekyelnyungmarts

## 2. Create selected CSV

This keeps the first `N_PER_CLASS` rows from each diagnostic class in `ham_test_cam_all_with_masks_grouped_by_class.csv`.

If the source CSV is already grouped by class, this gives a compact but balanced qualitative review set.

In [2]:
def make_first_n_per_class_csv(
    source_csv: Path,
    out_csv: Path,
    n_per_class: int,
    gt_col: str = "gt_label",
    require_masks: bool = True,
):
    df = pd.read_csv(source_csv, low_memory=False)

    required = ["image_rel_path", gt_col, "mask_rel_path"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {source_csv}: {missing}")

    if "image_id" not in df.columns:
        if "image" in df.columns:
            df["image_id"] = df["image"].astype(str).apply(lambda x: Path(x).stem)
        else:
            df["image_id"] = df["image_rel_path"].astype(str).apply(lambda x: Path(x).stem)

    if require_masks and "mask_found" in df.columns:
        df = df[df["mask_found"].fillna(0).astype(int) == 1].copy()

    class_order = ["MEL", "NV", "BKL", "BCC", "AKIEC", "DF", "VASC"]
    existing_order = [c for c in class_order if c in set(df[gt_col].astype(str))]
    remaining = [c for c in sorted(df[gt_col].dropna().astype(str).unique()) if c not in existing_order]
    class_order = existing_order + remaining

    selected_parts = []
    for cls in class_order:
        cls_df = df[df[gt_col].astype(str) == cls].copy()
        selected = cls_df.head(int(n_per_class)).copy()
        if len(selected) < n_per_class:
            print(f"[WARN] requested {n_per_class} for {cls}, found only {len(selected)}")
        selected_parts.append(selected)

    selected_df = pd.concat(selected_parts, axis=0).reset_index(drop=True)
    selected_df["class_order"] = selected_df[gt_col].map({c: i for i, c in enumerate(class_order)})
    selected_df = selected_df.sort_values(["class_order", "image_id"]).drop(columns=["class_order"])

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    selected_df.to_csv(out_csv, index=False)

    print("Saved:", out_csv)
    print("Rows:", len(selected_df))
    print("Class counts:")
    print(selected_df[gt_col].value_counts().reindex(class_order).fillna(0).astype(int))
    display(selected_df[["image_id", gt_col, "label", "image_rel_path", "mask_rel_path"]].head(20))

    return selected_df

selected_df = make_first_n_per_class_csv(
    source_csv=SOURCE_CSV,
    out_csv=SELECTED_CSV,
    n_per_class=N_PER_CLASS,
    gt_col=GT_COL,
    require_masks=True,
)

NUM_SAMPLES = len(selected_df)
print("NUM_SAMPLES:", NUM_SAMPLES)


[WARN] requested 20 for DF, found only 8
[WARN] requested 20 for VASC, found only 17
Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_first20_per_class_gap_cls_block_pdf.csv
Rows: 125
Class counts:
gt_label
MEL      20
NV       20
BKL      20
BCC      20
AKIEC    20
DF        8
VASC     17
Name: count, dtype: int64


,image_id,gt_label,label,image_rel_path,mask_rel_path
0,ISIC_0024459,MEL,4,images/ISIC_0024459.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
1,ISIC_0024571,MEL,4,images/ISIC_0024571.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
2,ISIC_0024624,MEL,4,images/ISIC_0024624.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
3,ISIC_0024640,MEL,4,images/ISIC_0024640.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
4,ISIC_0024756,MEL,4,images/ISIC_0024756.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
5,ISIC_0024886,MEL,4,images/ISIC_0024886.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
6,ISIC_0024967,MEL,4,images/ISIC_0024967.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
7,ISIC_0025105,MEL,4,images/ISIC_0025105.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
8,ISIC_0025132,MEL,4,images/ISIC_0025132.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
9,ISIC_0025234,MEL,4,images/ISIC_0025234.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...


NUM_SAMPLES: 125


## 3. Generate CAM panels

This creates one panel per scenario and image. Each scenario can use its own target block.

In [3]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def safe_name(name: str) -> str:
    out = str(name).lower()
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]", "{", "}", "+", "—"]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def resolved_block_index(target_block_index: int, n_blocks: int = 12) -> int:
    return n_blocks + target_block_index if target_block_index < 0 else target_block_index


def scenario_out_dir(scenario: dict) -> Path:
    return PANEL_ROOT / safe_name(scenario["short_name"])


def generate_comparison_panels(scenarios: list[dict], num_samples: int, dry_run: bool = False):
    panel_items = "rgb_gt_mask,gradcam_a,map_diff,finercam"

    for scenario in scenarios:
        scenario_name = scenario["name"]
        block_idx = int(scenario["target_block_index"])
        out_dir = scenario_out_dir(scenario)
        out_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(SELECTED_CSV),
            "--image_col", "image_rel_path",
            "--img_dir", str(IMG_DIR),
            "--gt_col", GT_COL,
            "--checkpoint", str(scenario["checkpoint"]),
            "--checkpoint_model_type", scenario.get("checkpoint_model_type", "panderm"),
            "--out_dir", str(out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--alpha", "0.8",
            "--panel_items", panel_items,
            "--mask_root", str(MASK_ROOT),
            "--mask_col", "mask_rel_path",
            "--target_block_index", str(block_idx),
            "--clinician_labels",
            "--model_display_name", scenario_name,
            # "--save_json",
            "--save_raw_cams",
        ]

        cmd += CLASS_ARGS
        cmd += COMPARE_ARGS

        print(
            f"\nGenerating panels: {scenario_name} | "
            f"target_block_index={block_idx} | resolved={resolved_block_index(block_idx)}"
        )
        run_command(cmd, dry_run=dry_run)


generate_comparison_panels(
    scenarios=SCENARIOS,
    num_samples=NUM_SAMPLES,
    dry_run=DRY_RUN,
)



Generating panels: GAP / mean pooling — block -10 | target_block_index=-10 | resolved=2

python -m scripts.generate_finer_cam_panderm --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_first20_per_class_gap_cls_block_pdf.csv --image_col image_rel_path --img_dir /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-HA075.pth --checkpoint_model_type panderm --out_dir /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/gap_cls_selected_blocks_pdf_first20_per_class_seed42/panels/gap_block_minus10 --num_samples 125 --method finercam --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,map_diff,finercam --mask_root /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --mask_col mask_rel_path --target_block_index -10 --clinician_labels --model_display_name 'GAP / mean pooling — block -10' --save_raw_cams --

## 4. Build comparison PDF

Each page shows one image. Each row shows one pooling/block scenario.

In [4]:
from PIL import Image, ImageDraw, ImageFont

PANEL_SUFFIX = "rgb_gt_mask_gradcam_a_map_diff_finercam"


def get_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        if path and Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()


FONT_TITLE = get_font(34, bold=True)
FONT_SUBTITLE = get_font(22, bold=False)
FONT_LABEL = get_font(23, bold=True)
FONT_SMALL = get_font(18, bold=False)


def image_id_to_stem(image_id_value: str) -> str:
    p = Path(str(image_id_value))
    stem = p.stem if p.suffix else p.name
    return stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def load_display_rows_for_pdf() -> pd.DataFrame:
    df = pd.read_csv(SELECTED_CSV, low_memory=False).head(NUM_SAMPLES).copy()
    if "image_id" not in df.columns:
        if "image_rel_path" in df.columns:
            df["image_id"] = df["image_rel_path"].apply(lambda x: Path(str(x)).stem)
        elif "image" in df.columns:
            df["image_id"] = df["image"].apply(lambda x: Path(str(x)).stem)
        else:
            raise ValueError("Need one of image_id, image_rel_path, or image columns.")
    return df


def find_panel_png(out_dir: Path, row: pd.Series) -> Path | None:
    candidates = []

    if "image_rel_path" in row and pd.notna(row["image_rel_path"]):
        candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image_id" in row and pd.notna(row["image_id"]):
        candidates.append(image_id_to_stem(row["image_id"]))
    if "image" in row and pd.notna(row["image"]):
        candidates.append(image_id_to_stem(row["image"]))

    for stem in dict.fromkeys(candidates):
        direct = out_dir / f"{stem}_{PANEL_SUFFIX}.png"
        if direct.exists():
            return direct
        matches = sorted(out_dir.glob(f"{stem}_*.png"))
        if matches:
            return matches[0]

    return None


def wrap_text(draw, text: str, font, max_width: int) -> list[str]:
    words = str(text).split()
    lines = []
    current = ""

    for word in words:
        test = f"{current} {word}".strip()
        bbox = draw.textbbox((0, 0), test, font=font)
        if bbox[2] - bbox[0] <= max_width:
            current = test
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return lines


def make_page_for_image(row: pd.Series, scenarios: list[dict]) -> Image.Image:
    page_width = 2300
    margin = 50
    label_width = 330
    gap = 18
    title_h = 130

    available_panel_width = page_width - 2 * margin - label_width - gap

    loaded_panels = []
    for scenario in scenarios:
        out_dir = scenario_out_dir(scenario)
        panel_path = find_panel_png(out_dir, row)
        if panel_path is None:
            loaded_panels.append((scenario, None, None))
            continue

        panel = Image.open(panel_path).convert("RGB")
        scale = available_panel_width / panel.width
        new_h = int(panel.height * scale)
        panel = panel.resize((available_panel_width, new_h), Image.Resampling.LANCZOS)
        loaded_panels.append((scenario, panel, panel_path))

    row_heights = []
    for _, panel, _ in loaded_panels:
        row_heights.append(panel.height if panel is not None else 260)

    page_height = title_h + margin + sum(row_heights) + gap * (len(row_heights) - 1) + margin
    page = Image.new("RGB", (page_width, page_height), "white")
    draw = ImageDraw.Draw(page)

    image_id = row.get("image_id", row.get("image_rel_path", "unknown"))
    gt = row.get(GT_COL, row.get("gt_label", "unknown"))
    title = f"GAP vs CLS block comparison: {image_id}"
    subtitle = f"Ground truth: {gt} | Same image across pooling and block choices"

    draw.text((margin, 30), title, fill="black", font=FONT_TITLE)
    draw.text((margin, 80), subtitle, fill=(60, 60, 60), font=FONT_SUBTITLE)

    y = title_h
    for scenario, panel, panel_path in loaded_panels:
        row_h = panel.height if panel is not None else 260

        block_idx = int(scenario["target_block_index"])
        resolved = resolved_block_index(block_idx)
        label = scenario["name"]
        label_lines = wrap_text(draw, label, FONT_LABEL, label_width - 10)
        label_x = margin
        label_y = y + 25

        for line in label_lines:
            draw.text((label_x, label_y), line, fill="black", font=FONT_LABEL)
            label_y += 32

        draw.text(
            (label_x, label_y + 10),
            f"target {block_idx} → block {resolved}",
            fill=(80, 80, 80),
            font=FONT_SMALL,
        )

        if panel is None:
            box_x = margin + label_width + gap
            box_y = y
            draw.rectangle([box_x, box_y, box_x + available_panel_width, box_y + row_h], outline=(180, 180, 180), width=2)
            draw.text((box_x + 30, box_y + 80), "Missing panel PNG", fill=(160, 0, 0), font=FONT_LABEL)
        else:
            page.paste(panel, (margin + label_width + gap, y))

        y += row_h + gap

    return page


def build_comparison_pdf():
    rows = load_display_rows_for_pdf()

    pages = []
    for _, row in rows.iterrows():
        page = make_page_for_image(row, SCENARIOS)
        pages.append(page)

    if not pages:
        raise RuntimeError("No pages generated.")

    PDF_OUT.parent.mkdir(parents=True, exist_ok=True)
    pages[0].save(PDF_OUT, save_all=True, append_images=pages[1:], resolution=150.0)

    config = {
        "qual_seed": QUAL_SEED,
        "n_per_class": N_PER_CLASS,
        "num_samples": NUM_SAMPLES,
        "source_csv": str(SOURCE_CSV),
        "selected_csv": str(SELECTED_CSV),
        "pdf_out": str(PDF_OUT),
        "out_root": str(OUT_ROOT),
        "scenarios": [
            {
                "name": s["name"],
                "short_name": s["short_name"],
                "checkpoint": str(s["checkpoint"]),
                "checkpoint_model_type": s.get("checkpoint_model_type", "panderm"),
                "target_block_index": s["target_block_index"],
                "resolved_block_index": resolved_block_index(int(s["target_block_index"])),
            }
            for s in SCENARIOS
        ],
        "class_args": CLASS_ARGS,
        "compare_args": COMPARE_ARGS,
    }
    CONFIG_OUT.write_text(json.dumps(config, indent=2))

    print("Saved PDF:", PDF_OUT)
    print("Saved config:", CONFIG_OUT)


build_comparison_pdf()


Saved PDF: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/gap_cls_selected_blocks_pdf_first20_per_class_seed42/gap_cls_selected_blocks_first20_per_class_seed42.pdf
Saved config: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/gap_cls_selected_blocks_pdf_first20_per_class_seed42/gap_cls_selected_blocks_config_first20_per_class_seed42.json


## 5. Quick checks

Run this after generating the PDF to confirm that every scenario created the expected number of panels.

In [5]:
for scenario in SCENARIOS:
    out_dir = scenario_out_dir(scenario)
    pngs = sorted(out_dir.glob("*.png"))
    metas = sorted(out_dir.glob("*_meta.json"))
    raw_cams = sorted((out_dir / "raw_cams").glob("**/*.npy")) if (out_dir / "raw_cams").exists() else []
    print("\n" + scenario["name"])
    print("  out_dir:", out_dir)
    print("  target_block_index:", scenario["target_block_index"], "resolved:", resolved_block_index(int(scenario["target_block_index"])))
    print("  png panels:", len(pngs), "/ expected", NUM_SAMPLES)
    print("  meta files:", len(metas))
    print("  raw CAM arrays:", len(raw_cams))
    if pngs[:3]:
        for p in pngs[:3]:
            print("   ", p.name)

print("\nPDF_OUT:", PDF_OUT)



GAP / mean pooling — block -10
  out_dir: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/gap_cls_selected_blocks_pdf_first20_per_class_seed42/panels/gap_block_minus10
  target_block_index: -10 resolved: 2
  png panels: 125 / expected 125
  meta files: 0
  raw CAM arrays: 1000
    ISIC_0024326_rgb_gt_mask_gradcam_a_map_diff_finercam.png
    ISIC_0024344_rgb_gt_mask_gradcam_a_map_diff_finercam.png
    ISIC_0024353_rgb_gt_mask_gradcam_a_map_diff_finercam.png

GAP / mean pooling — last block
  out_dir: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/gap_cls_selected_blocks_pdf_first20_per_class_seed42/panels/gap_last_block
  target_block_index: -1 resolved: 11
  png panels: 125 / expected 125
  meta files: 0
  raw CAM arrays: 1000
    ISIC_0024326_rgb_gt_mask_gradcam_a_map_diff_finercam.png
    ISIC_0024344_rgb_gt_mask_gradcam_a_map_diff_finercam.png
    ISIC_0024353_rgb_gt_mask_gradcam_a_map_diff_finercam.png

GAP 2 / mean pooling — block -10
  out_dir: /Users/